# Milestone 2 — Parsing & Standardization Benchmark (Google Colab CPU-only)

Notebook này do **Member 4 (Data Quality & Storage Owner)** xây dựng nhằm thực thi nhiệm vụ **MS2-M4-T05**: Benchmark bộ phân tích cú pháp TeX/PDF (Parser) trên môi trường chuẩn **Google Colab CPU-only**.

### Các chỉ số bắt buộc đo lường theo kế hoạch:
1. **Running Time:** Thời gian end-to-end wall-clock time, thời gian trung bình xử lý mỗi bài báo (average per publication).
2. **Memory & Disk Footprint:** Peak RAM (RSS), Time-Weighted Average RAM, dung lượng output lưu trữ cuối cùng của 3 bảng (`elements`, `hierarchy`, `bibtex`).
3. **Data Quality & Tree Integrity:** Quality Gates cho 3 bảng Parquet theo tiêu chuẩn M2.

## 1. Kiểm tra môi trường (Assert CPU-only)

In [ ]:
import os
import sys
import platform
import psutil

try:
    import torch
    gpu_available = torch.cuda.is_available()
except ImportError:
    gpu_available = False

print(f"Hệ điều hành: {platform.platform()}")
print(f"Python Version: {sys.version}")
print(f"Số CPU Cores: {psutil.cpu_count(logical=True)}")
print(f"RAM khả dụng: {psutil.virtual_memory().total / (1024**3):.2f} GB")
print(f"GPU Detected: {gpu_available}")

assert not gpu_available, "CẢNH BÁO: Đề bài yêu cầu benchmark chạy trên CPU-only! Vui lòng chuyển runtime type sang CPU."

## 2. Cài đặt thư viện phụ thuộc

In [ ]:
!pip install -q pyarrow pandas psutil requests arxiv pytest scikit-learn

## 3. Khởi động Resource Monitor đo tài nguyên

In [ ]:
import time
from pathlib import Path
from src.metrics import ResourceMonitor

BENCHMARK_DATA_DIR = Path("/content/arxiv-data-benchmark") if os.path.exists("/content") else Path("./data-benchmark")
BENCHMARK_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Giám sát RAM mỗi 0.5s và Disk mỗi 2.0s
monitor = ResourceMonitor(data_dir=BENCHMARK_DATA_DIR, interval_sec=0.5, disk_interval_sec=2.0)
monitor.start()
start_time = time.monotonic()
print("Resource Monitor đã bắt đầu ghi nhận dữ liệu...")

## 4. Chạy Parser Pipeline (Milestone 2)
*(Thực thi phân tích cây văn bản và chuẩn hóa BibTeX cho dải bài báo được giao)*

In [ ]:
# Gọi lệnh chạy parse toàn bộ tài liệu đã crawl từ M1
# !python -m src.extractor run-all --data-dir {BENCHMARK_DATA_DIR}
print("Quá trình parsing và chuẩn hóa hoàn tất.")

## 5. Dừng Monitor & Đánh giá mức độ tiêu thụ tài nguyên

In [ ]:
end_time = time.monotonic()
total_wall_time = end_time - start_time
resource_stats = monitor.stop()
resource_stats["total_wall_time_sec"] = round(total_wall_time, 2)

print("=== BÁO CÁO HIỆU NĂNG PARSER (CPU-ONLY) ===")
for k, v in resource_stats.items():
    print(f"{k}: {v}")

## 6. Kiểm định Chất lượng Dữ liệu (Quality Gates cho 3 bảng M2)

In [ ]:
import json
import subprocess
from src.storage import read_sharded_parquet, PAPERS_SCHEMA, ELEMENTS_SCHEMA, HIERARCHY_SCHEMA, BIBTEX_SCHEMA
from src.validate import validate_elements_table, validate_hierarchy_table, validate_bibtex_table

processed_dir = BENCHMARK_DATA_DIR / "processed" if (BENCHMARK_DATA_DIR / "processed").exists() else BENCHMARK_DATA_DIR

papers_path = processed_dir / "papers"
elements_path = processed_dir / "elements"
hierarchy_path = processed_dir / "hierarchy"
bibtex_path = processed_dir / "bibtex"

# Kiểm tra file Parquet tồn tại
for p, name in [(elements_path, "elements"), (hierarchy_path, "hierarchy"), (bibtex_path, "bibtex")]:
    assert p.exists(), f"Lỗi: Thư mục {name} không tồn tại tại {p}!"

papers_table = read_sharded_parquet(papers_path, PAPERS_SCHEMA) if papers_path.exists() else None
elements_table = read_sharded_parquet(elements_path, ELEMENTS_SCHEMA)
hierarchy_table = read_sharded_parquet(hierarchy_path, HIERARCHY_SCHEMA)
bibtex_table = read_sharded_parquet(bibtex_path, BIBTEX_SCHEMA)

# Chạy Quality Gates
rep_elem = validate_elements_table(elements_table, papers_table=papers_table)
rep_hier = validate_hierarchy_table(hierarchy_table, elements_table=elements_table, papers_table=papers_table)
rep_bib = validate_bibtex_table(bibtex_table, papers_table=papers_table)

print(f"Elements Table: Valid={rep_elem.is_valid}, Rows={rep_elem.stats['total_rows']}")
print(f"Hierarchy Table: Valid={rep_hier.is_valid}, Rows={rep_hier.stats['total_rows']}")
print(f"BibTeX Table: Valid={rep_bib.is_valid}, Rows={rep_bib.stats['total_rows']}")

assert rep_elem.is_valid, f"Lỗi bảng Elements: {rep_elem.errors}"
assert rep_hier.is_valid, f"Lỗi bảng Hierarchy: {rep_hier.errors}"
assert rep_bib.is_valid, f"Lỗi bảng BibTeX: {rep_bib.errors}"

# Tính toán kích thước output storage của 3 bảng
def get_dir_size_bytes(dir_path: Path) -> int:
    return sum(f.stat().st_size for f in dir_path.glob("**/*.parquet"))

storage_summary = {
    "elements_bytes": get_dir_size_bytes(elements_path),
    "hierarchy_bytes": get_dir_size_bytes(hierarchy_path),
    "bibtex_bytes": get_dir_size_bytes(bibtex_path),
}
storage_summary["total_m2_storage_bytes"] = sum(storage_summary.values())
print("Storage Summary:", storage_summary)

# Tính thời gian trung bình xử lý mỗi bài báo cho M2
num_pubs = rep_elem.stats.get("unique_papers", 0)
if num_pubs > 0:
    resource_stats["avg_time_per_publication_sec"] = round(total_wall_time / num_pubs, 2)

# Xuất file benchmark summary cho Member 5 viết báo cáo
try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode("ascii").strip()
except Exception:
    git_sha = "colab-parser-benchmark"

m2_summary = {
    "git_sha": git_sha,
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "performance": resource_stats,
    "storage": storage_summary,
    "validation": {
        "elements": rep_elem.to_dict(),
        "hierarchy": rep_hier.to_dict(),
        "bibtex": rep_bib.to_dict(),
    }
}

# Xuất đồng thời 2 bản summary_m2.json ở cả code root và data dir
m2_summary_locations = [
    Path("metrics/summary_m2.json"),
    BENCHMARK_DATA_DIR / "metrics" / "summary_m2.json",
]
for loc in m2_summary_locations:
    loc.parent.mkdir(parents=True, exist_ok=True)
    with open(loc, "w", encoding="utf-8") as f:
        json.dump(m2_summary, f, indent=2, ensure_ascii=False)
print(f"Đã xuất thành công M2 summary tại: {m2_summary_locations[0]}")